# Scene batch

klein draws one complete vertical scene per job, conditioned on costume reference pictures.
Every number in the report is measured; a failed picture is recorded, not hidden.


In [ ]:
import os, sys, json, time, subprocess, pathlib, hashlib, tarfile, traceback, uuid, urllib.request

RUN_ID = globals().get("RUN_ID", "manual-" + uuid.uuid4().hex)
RUN_TOKEN = globals().get("RUN_TOKEN", "manual")
COMMIT_SHA = globals().get("COMMIT_SHA", "unknown")
WORKFLOW_RUN_ID = globals().get("WORKFLOW_RUN_ID", "local")

OUT = pathlib.Path(os.environ.get("STICKMAN_OUT_DIR", "/kaggle/working/outputs"))  # override only for unit tests
WORK = pathlib.Path(os.environ.get("STICKMAN_WORK_DIR", "/tmp/scene_batch"))
LOGS = OUT / "logs"
for d in (OUT, LOGS, WORK):
    d.mkdir(parents=True, exist_ok=True)

REPO = "damarlanandu2005-jpg/stickman-motion-studio"
FALLBACK_REF = "test/image-to-3d-objects"
REFDIR = "animation/reference_art/"
REFS = {}

KLEIN_ID, KLEIN_REVISION = "black-forest-labs/FLUX.2-klein-4B", "e7b7dc27f91deacad38e78976d1f2b499d76a294"
KLEIN_PIP = ["diffusers==0.42.1", "transformers==5.22.1", "huggingface-hub==1.33.0", "accelerate", "pillow"]
CACHE_ENV = {"HF_HOME": "/tmp/hf_home", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}

JOBS = {
 "bg_v3_street_barrier": {
  "refs": [],
  "prompt": "A grey city street in daylight with grey concrete road barriers across the middle and a yellow tape stretched along them, red brick buildings far behind, wide empty asphalt in front. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8000,
  "width": 576,
  "height": 1024
 },
 "bg_v3_garage": {
  "refs": [],
  "prompt": "The inside of a police station garage: a grey roller shutter door, two grey metal lockers on the right, a fluorescent ceiling light, light grey walls, concrete floor. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8001,
  "width": 576,
  "height": 1024
 },
 "bg_v3_street_corner": {
  "refs": [],
  "prompt": "A city street corner in daylight: red brick apartment buildings with shop fronts and awnings, a grey sidewalk, a white zebra crossing in the foreground, blue sky. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8002,
  "width": 576,
  "height": 1024
 },
 "bg_v3_hospital": {
  "refs": [],
  "prompt": "A hospital examination room: pale green and white walls, a height measuring chart on the wall, a round wall clock, a pink hospital bed on the left, light tiled floor. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8003,
  "width": 576,
  "height": 1024
 },
 "bg_v3_bulletin": {
  "refs": [],
  "prompt": "A tan painted wall with a large empty cork board on it, a wooden bench on the right side, grey floor, indoor lighting. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8004,
  "width": 576,
  "height": 1024
 },
 "bg_v3_police_steps": {
  "refs": [],
  "prompt": "The front of a grey stone government building with tall pillars, a wide staircase of steps in front, an empty blank sign panel above the double doors, blue sky with one cloud. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8005,
  "width": 576,
  "height": 1024
 },
 "bg_v3_night_study": {
  "refs": [],
  "prompt": "A small room at night: a big window showing a night sky with stars and a crescent moon, a wooden desk on the right with a desk lamp glowing yellow, dark blue walls. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8006,
  "width": 576,
  "height": 1024
 },
 "bg_v3_dispatch": {
  "refs": [],
  "prompt": "A police radio dispatch room: a big city street map on the wall, a desk with a radio set and a computer monitor, dark blue-grey walls. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8007,
  "width": 576,
  "height": 1024
 },
 "bg_v3_warehouse": {
  "refs": [],
  "prompt": "A big dark empty warehouse with steel beams, tall windows letting in beams of light, wooden crates stacked at the back, concrete floor. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8008,
  "width": 576,
  "height": 1024
 },
 "bg_v3_classroom": {
  "refs": [],
  "prompt": "A police academy classroom: a green chalkboard, wooden desks and a teacher lectern, a flag on a pole on the left, a round wall clock, light grey walls. Keep the lower third of the picture as empty floor or ground. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8009,
  "width": 576,
  "height": 1024
 },
 "bg_v3_skyline": {
  "refs": [],
  "prompt": "A city skyline in silhouette at sunset, pink and orange sky, dark water in the foreground reflecting the skyline, a small boat at the bottom. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8010,
  "width": 576,
  "height": 1024
 },
 "ins_traffic_grey": {
  "refs": [],
  "prompt": "A plain light grey background. One tall black traffic light on a pole in the middle, all three lamps switched off and dark grey. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8011,
  "width": 576,
  "height": 1024
 },
 "ins_cards_orange": {
  "refs": [],
  "prompt": "A flat orange background with a big darker orange circle behind. Five playing-card shaped colour cards fanned out in the middle: red, green, blue, yellow, orange. The cards are blank. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8012,
  "width": 576,
  "height": 1024
 },
 "ins_calendar": {
  "refs": [],
  "prompt": "A flat pale beige background. One tear-off wall calendar with a red header and two metal rings at the top, a blank white page. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8013,
  "width": 576,
  "height": 1024
 },
 "ins_blueprints_orange": {
  "refs": [],
  "prompt": "A flat orange background with a darker orange circle. Three blue technical blueprint sheets lying slightly rotated over each other, white line drawings of electronic parts on blue grid paper, no writing. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8014,
  "width": 576,
  "height": 1024
 },
 "ins_blueprint_battery": {
  "refs": [],
  "prompt": "A flat blue blueprint background with a fine grid. White outline drawing of four battery cells in a row inside a rectangular frame, no writing. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8015,
  "width": 576,
  "height": 1024
 },
 "ins_blueprint_circuit": {
  "refs": [],
  "prompt": "A flat blue blueprint background with a fine grid. White outline drawing of a circuit board with chips and connecting lines inside a rectangular frame, no writing. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8016,
  "width": 576,
  "height": 1024
 },
 "ins_wires_straight_grey": {
  "refs": [],
  "prompt": "A flat dark grey background. Five thick grey wires running horizontally across the middle, held at the left and right by dark metal clamps. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8017,
  "width": 576,
  "height": 1024
 },
 "ins_wires_cut_red_pliers": {
  "refs": [],
  "prompt": "A flat dark grey background. A bundle of colourful wires, red, green, blue, yellow, orange and white, running left to right, with red-handled wire cutters closing on the blue wire at the right. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8018,
  "width": 576,
  "height": 1024
 },
 "ins_wires_knot_grey": {
  "refs": [],
  "prompt": "A flat dark grey background. A bundle of thick grey wires twisted together in a tight knot in the middle, held by dark metal clamps left and right. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8019,
  "width": 576,
  "height": 1024
 },
 "ins_timer_dark": {
  "refs": [],
  "prompt": "A flat dark grey background. A black bomb timer box with a blank dark display window at the top, three yellow and orange battery cells and red, green, yellow and blue wires connected below it. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8020,
  "width": 576,
  "height": 1024
 },
 "ins_timer_red": {
  "refs": [],
  "prompt": "A flat dark red background. A black bomb timer box with a blank dark display window at the top, three orange battery cells and red, green and blue wires connected below it. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8021,
  "width": 576,
  "height": 1024
 },
 "ins_explosion": {
  "refs": [],
  "prompt": "A big orange and yellow cartoon explosion fireball with dark grey smoke clouds, in front of a grey city street, bright flames filling the frame. Flat 2D vector cartoon, thick clean dark outlines, muted limited colour palette, soft flat shading. One complete vertical 9:16 illustration. There is no character, no person, no face and no eyes anywhere in the picture, only the objects and the place. No text, no letters, no numbers, no captions, no signs with writing, no watermark, no panel borders.",
  "seed": 8022,
  "width": 576,
  "height": 1024
 }
}
STEPS, GUIDANCE = 4, 1.0

REPORT = {
    "run_token": RUN_TOKEN, "run_id": RUN_ID, "commit_sha": COMMIT_SHA, "workflow_run_id": WORKFLOW_RUN_ID,
    "status": "running", "error_message": None, "poses": list(JOBS), "seeds": [0], "reference": REFDIR,
    "environment": {}, "klein": {}, "files": [],
}
STARTED = time.monotonic()


In [ ]:
KLEIN_STAGE = 'import json, sys, time, hashlib, traceback\nfrom pathlib import Path\ncfg = json.loads(Path(sys.argv[1]).read_text())\nres = {"stage": "klein", "status": "running", "error": None, "pictures": []}\ntry:\n    import torch\n    from PIL import Image\n    from huggingface_hub import snapshot_download\n    from diffusers import Flux2KleinPipeline\n    t = time.monotonic(); path = snapshot_download(cfg["model_id"], revision=cfg["revision"])\n    res["download_seconds"] = round(time.monotonic() - t, 1)\n    t = time.monotonic(); pipe = Flux2KleinPipeline.from_pretrained(path, torch_dtype=torch.float16)\n    pipe.enable_model_cpu_offload(gpu_id=0); res["load_seconds"] = round(time.monotonic() - t, 1)\n    out = Path(cfg["out_dir"]); out.mkdir(parents=True, exist_ok=True)\n    for name, job in cfg["jobs"].items():\n        item = {"pose": name, "seed": job["seed"], "status": "running"}\n        res["pictures"].append(item)\n        try:\n            refs = []\n            for r in job["refs"]:\n                im = Image.open(Path(cfg["refdir"]) / r).convert("RGB"); im.thumbnail((1024, 1024)); refs.append(im)\n            torch.cuda.synchronize(); t = time.monotonic()\n            kw = {"image": refs} if refs else {}  # object-only pictures use no reference, so no mascot face leaks onto objects\n            img = pipe(prompt=job["prompt"], width=job["width"], height=job["height"], num_inference_steps=cfg["steps"],\n                       guidance_scale=cfg["guidance"], generator=torch.Generator("cpu").manual_seed(job["seed"]), **kw).images[0]\n            torch.cuda.synchronize(); secs = round(time.monotonic() - t, 2)\n            p = out / f"pose_{name}.png"; img.save(p)\n            item.update({"file": p.name, "seconds": secs, "width": img.width, "height": img.height,\n                         "sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "status": "success"})\n            print("klein drew", p.name, "in", secs, "s", flush=True)\n        except Exception as exc:  # one bad picture must not stop the rest\n            item.update({"status": "failed", "error": f"{type(exc).__name__}: {exc}"[:500]}); traceback.print_exc()\n            try:\n                torch.cuda.empty_cache()\n            except Exception:\n                pass\n    res["peak_vram_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)\n    res["status"] = "success" if all(i["status"] == "success" for i in res["pictures"]) else "partial"\nexcept Exception as exc:\n    res["status"] = "failed"; res["error"] = f"{type(exc).__name__}: {exc}"[:3000]; traceback.print_exc()\nPath(cfg["result"]).write_text(json.dumps(res, indent=2))\n'


In [ ]:
def run_logged(name, cmd, timeout):
    log = LOGS / f"{name}.log"
    t = time.monotonic()
    try:
        with log.open("a") as fh:
            fh.write("$ " + " ".join(map(str, cmd)) + "\n"); fh.flush()
            code = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, timeout=timeout, env={**os.environ, **CACHE_ENV}).returncode
        outcome = "ok" if code == 0 else f"exit {code}"
    except subprocess.TimeoutExpired:
        code, outcome = None, f"timed out after {timeout} s"
    print(f"[{name}] {outcome} in {round(time.monotonic() - t, 1)} s", flush=True)
    return code == 0


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def fetch_reference():
    ref = COMMIT_SHA if len(COMMIT_SHA) == 40 else FALLBACK_REF
    tgz = WORK / "repo.tar.gz"
    urllib.request.urlretrieve(f"https://codeload.github.com/{REPO}/tar.gz/{ref}", tgz)
    with tarfile.open(tgz) as tf:
        tf.extractall(WORK / "repo", filter="data")
    folder = next((WORK / "repo").iterdir()) / REFDIR
    for name, digest in REFS.items():
        if sha256_of(folder / name) != digest:
            raise RuntimeError(f"reference picture {name} does not match the approved file (hash differs)")
    return folder


def environment():
    import torch
    env = REPORT["environment"]
    env["python"] = sys.version.split()[0]
    env["torch"] = torch.__version__
    env["cuda_available"] = bool(torch.cuda.is_available())
    env["gpu_names"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
    if not env["cuda_available"]:
        raise RuntimeError("CUDA is not available: enable the GPU accelerator.")


In [ ]:
try:
    environment()
    reference = fetch_reference()
    if not run_logged("install_klein", [sys.executable, "-m", "pip", "install", "--no-cache-dir", *KLEIN_PIP], 1200):
        raise RuntimeError("klein dependencies failed to install; see logs/install_klein.log")
    script, cfg, result = WORK / "klein_stage.py", WORK / "klein_stage.json", OUT / "result_klein.json"
    script.write_text(KLEIN_STAGE, encoding="utf-8")
    cfg.write_text(json.dumps({"model_id": KLEIN_ID, "revision": KLEIN_REVISION, "refdir": str(reference), "jobs": JOBS,
                               "steps": STEPS, "guidance": GUIDANCE,
                               "result": str(result), "out_dir": str(OUT)}), encoding="utf-8")
    run_logged("klein", [sys.executable, str(script), str(cfg)], 3000)
    if not result.exists():
        raise RuntimeError("klein wrote no result; see logs/klein.log")
    REPORT["klein"] = json.loads(result.read_text(encoding="utf-8"))
    REPORT["files"] = [p["file"] for p in REPORT["klein"]["pictures"] if p.get("status") == "success"]
    if REPORT["klein"]["status"] == "failed":
        raise RuntimeError("klein failed: " + str(REPORT["klein"].get("error")))
    REPORT["status"] = "completed"  # per-picture success is in REPORT["klein"]["pictures"]
except Exception as exc:
    REPORT["status"] = "failed"
    REPORT["error_message"] = f"{type(exc).__name__}: {exc}"
    traceback.print_exc()
finally:
    REPORT["notebook_duration_seconds"] = round(time.monotonic() - STARTED, 1)
    (OUT / "poses_report.json").write_text(json.dumps(REPORT, indent=2), encoding="utf-8")
    print(json.dumps({k: REPORT[k] for k in ("status", "error_message", "files", "notebook_duration_seconds")}, indent=2))


In [ ]:
assert REPORT["status"] == "completed", "Poses did not complete: " + str(REPORT["error_message"])
print("Drew", len(REPORT["files"]), "of", len(JOBS), "pictures")
